In [7]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 5  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line): 

John Mandalas

Mohammadreza Babaeizadeh

Chloe Pang

Pranav Sudev

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

## Variables

Global variables that can be configured.


In [ ]:
# Client Configurations
client_a_budget = 1000000
client_b_budget = 750000

min_port_size = 10
max_port_size = 20

max_weight_each = 0.15
max_weight_sector = 0.40

# Stock eligibility
csv_file_attach = "Tickers_Example.csv"
"""CHANGE THE CSV FILE NAME BEFORE SUBMIT TO Tickers.csv"""
min_avg_d_vol = 5000

start_date = "2025-10-01"
end_date = "2026-10-01"

purchase_date = "2026-11-27"
purchase_end_date = "2026-11-28" # Downloading close price.

min_trade_d_per_m = 18

# Purchase and transaction fees in USD
fee_cap_per_stock = 2.15
fee_per_share = 0.001 

In [11]:
# Gets all the list of tickers from the csv file and returns them as a pandas dataframe
# Parameters: file_name (str) - the name of the csv file containing the tickers
# Returns: pandas dataframe containing the tickers
def get_listof_tickers(file_name):
    listof_tickers = pd.read_csv(file_name, header=None)
    listof_tickers = listof_tickers.drop_duplicates().reset_index(drop=True)
    return listof_tickers


# Normalizes the list of tickers to ensure they fit with the assignment criteria
# Parameters: tickers (pandas dataframe) - the dataframe containing the tickers to normalize
# Returns: pandas dataframe containing the normalized tickers
def normalize_listof_tickers(tickers):
    stock_data = []

    valid_exchanges = ["NMS", "NYQ", "ASE", "TOR", "VAN", "NEO"]# TBDTBDTBD
    
    # Get data needed for each ticker in the dataframe.
    for ticker in tickers[0]:
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            history = stock.history(start="2025-10-01", end="2026-10-01", auto_adjust=False) # Last trading date checked: 2026-09-30

            # Skip stocks without historical data
            if history.empty:
                print("Not Valid Ticker (no data):", ticker)
                continue

            # Check if the stock meets the basic requirements
            valid_stock = (info.get("quoteType") == "EQUITY" 
                           and info.get("currency") in ["CAD", "USD"] 
                           and info.get("exchange") in valid_exchanges)
           
            # Counting trading days in each calendar month
            month_counts = {}

            # Go through all dates and count trading days for each month
            for date in history.index:
                month = date.month

                if month in month_counts:
                    month_counts[month] += 1
                else:
                    month_counts[month] = 1

            lowest_trading_day = min(month_counts.values())

            # Calculate average daily trading volume
            total_volume = 0
            total_days = 0
            
            for date, row in history.iterrows():
                month = date.month
                
                # Only include months with at least 18 trading days
                if month_counts[month] >= min_trade_d_per_m: 
                    total_volume += row["Volume"]
                    total_days += 1
            
            if total_days > 0:
                average_volume = total_volume / total_days
            else:
                average_volume = 0

            # Creating a dataframe containing information needed in further functions
            if valid_stock and average_volume >= min_avg_d_vol:
                stock_data.append({
                    "Ticker": ticker,
                    "Stock's Name": info.get("longName"),
                    "Exchange": info.get("exchange"),
                    "Currency": info.get("currency"),
                    "Ticker Type": info.get("quoteType"),
                    "Sector": info.get("sector"),
                    "Monthly Trading Days": lowest_trading_day,
                    "Average Volume": average_volume
            })

        # Mark the stock as invalid if an error occurs
        except Exception as e:
            print("Not Valid Ticker:", ticker, "-", e)

    return pd.DataFrame(stock_data)


# Use the new list that only contains the valid data (Not necessary to produce invalid data)
# Returns: a dataframe cointaining the {TICKER}_CLOSE and {TICKER}_ADJ_CLOSE for each ticker in the list of tickers, for the given start and end dates
def create_dataframe(listof_tickers, start_date, end_date):
    stock_prices = pd.DataFrame()
    for ticker in listof_tickers["Ticker"]: # Change [0] to ["Ticker"] just to get the valid stocks price
        stock = yf.Ticker(ticker)
        history = stock.history(start=start_date, end=end_date,auto_adjust=False)

        if not history.empty:
            history.index = history.index.tz_localize(None)
            prices = history[["Close", "Adj Close"]].copy() #Add other categories to here
            prices.columns = [ticker + "_Close", ticker + "_Adj_Close"]
            stock_prices = pd.concat([stock_prices, prices], axis=1,sort=True)

    return stock_prices


# Gets the USD to CAD exchange rate between two dates using yfinance
# Parameters: start_date (str) - the start date to get the exchange rate for in the format 'YYYY-MM-DD'
# end_date (str) - the end date to get the exchange rate for in the format 'YYYY-MM-DD'
# Returns: float - the USD to CAD exchange rate for the given date
def get_USD_to_CAD_exchange_rate(start_date, end_date):
    # Get the USD to CAD exchange rate for the given date using yfinance
    exchange_rate = yf.download('CAD=X', start=start_date, end=end_date)
    return exchange_rate['Close'].squeeze("columns")

# Buys a ticker and adds it to the portfolio, including fees
# Parameters: ticker_name (str) - the ticker to buy, dollars (float) - the amount of CAD to spend including fees,
# portfolio (pandas dataframe) - the portfolio to add the ticker to, date (str) - the purchase date 'YYYY-MM-DD'
# Returns: num_shares (float) - the number of shares bought
def buy_ticker(ticker_name, dollars, portfolio, date):
    rate = exchange_rates.loc[date]
    price = stock_prices.loc[date, ticker_name + "_Close"]

    # Convert USD prices to CAD
    if normalized_tickers.set_index("Ticker").loc[ticker_name, "Currency"] == "USD":
        price = price * rate

    # Per-share fee case: dollars = shares * price + shares * 0.001 * rate
    num_shares = dollars / (price + rate * 0.001)

    # If the per-share fee would be more than the flat fee, the flat fee applies
    if calculate_transaction_fees(num_shares, rate) < num_shares * rate * 0.001:
        num_shares = (dollars - rate * 2.15) / price

    portfolio.loc[ticker_name, "Shares"] = num_shares
    return num_shares


# Calculates volatility of the whole portfolio over a given time period using the standard deviation of the returns
# Parameters: daily_returns (pandas series), the series that contains the daily returns of the portfolio over a given time period
# Returns: the volatility of the portfolio 
def calculate_volatility(daily_returns):
    return daily_returns.std()

# Calculate transaction fees for buying a ticker
# Parameters: shares (float), the number of shares to buy
# exchange_rate (float), CAD per US$1, the exchange rate for the given date
# Returns: the transaction fees for the stock purchase (in CAD)
def calculate_transaction_fees(shares, exchange_rate):
    fees = min(exchange_rate * 2.15, shares * exchange_rate * 0.001)
    return fees

# Calculates the value of a portfolio on a given date in CAD
# Parameters: portfolio (pandas dataframe) - the portfolio, date (str) - the date 'YYYY-MM-DD'
# Returns: the total value of the portfolio in CAD
def portfolio_value(portfolio, date):
    rate = exchange_rates.loc[date]
    total = 0
    for ticker in portfolio.index:
        price = stock_prices.loc[date, ticker + "_Close"]
        if normalized_tickers.set_index("Ticker").loc[ticker, "Currency"] == "USD":
            price = price * rate
        total += portfolio.loc[ticker, "Shares"] * price
    return total


# Calculates the portfolio's return over the comparison period
# Parameters: portfolio (pandas dataframe) - the portfolio, start_date (str) - start of the period, end_date (str) - end of the period
# Returns: the portfolio's return as a decimal (0.05 = 5%)
def calculate_portfolio_return(portfolio, start_date, end_date):
    return portfolio_value(portfolio, end_date) / portfolio_value(portfolio, start_date) - 1


# Convert USD stock prices to CAD using the exchange rate for each given date
# Parameters: stock_prices (pandas dataframe), the dataframe that contains the stock prices for each ticker in USD
# exchange_rates (pandas series), the series that contains the exchange rates for each date
# Returns: the pandas dataframe that contains the stock prices for each ticker in CAD
def convert_usd_to_cad(stock_prices, exchange_rates, normalized_tickers):
    converted_prices = stock_prices.copy()
    for i, stock in normalized_tickers.iterrows():
        ticker = stock["Ticker"]
        if stock["Currency"] == "USD": 
            for column in [ticker + "_Close", ticker + "_Adj_Close"]:
                if column in converted_prices.columns:
                    converted_prices[column] = converted_prices[column] * exchange_rates
    return converted_prices


# Calculate the historical portfolio value over a given time period (CAD)
# Parameters: stock_prices (pandas dataframe), the dataframe that contains the stock prices for each ticker (CAD)
# portfolio (pandas dataframe), the dataframe that contains the tickers and number of shares in the portfolio
# Returns: the pandas series that contains the historical portfolio value over the given time period
def calculate_portfolio_value(stock_prices, portfolio):
    portfolio_values = pd.Series(0.0, index=stock_prices.index)
    for ticker in portfolio.index:
        shares = portfolio.loc[ticker, "Shares"]
        if ticker + "_Close" in stock_prices.columns:
            portfolio_values += stock_prices[ticker + "_Close"] * shares
    return portfolio_values



# Calculate the daily returns of the portfolio over a given time period
# Parameters: portfolio_values (pandas series), the series that contains the historical portfolio value over the given time period
# Returns: the pandas series that contains the daily returns of the portfolio over the given time period
def calculate_daily_returns(portfolio_values):
    daily_returns = portfolio_values.pct_change().dropna()
    return daily_returns

# Calling ticker list
listof_tickers = get_listof_tickers(csv_file_attach)
display(listof_tickers)

# Calling valid ticker 
normalized_tickers = normalize_listof_tickers(listof_tickers)
display(normalized_tickers)

# Calling create_dataframe for Adj and Close price
stock_prices = create_dataframe(normalized_tickers, start_date, end_date)
stock_prices = stock_prices.ffill()   # holiday: use the last close
display(stock_prices)

# Calling the exchange_rates USD CAD
exchange_rates = get_USD_to_CAD_exchange_rate(start_date, end_date)
exchange_rates = exchange_rates.reindex(stock_prices.index).ffill()
display(exchange_rates)

,0
0,AAPL
1,ABBV
2,ABT
3,ACN
4,AGN
5,AIG
6,AMZN
7,AXP
8,BA
9,BAC


$AGN: possibly delisted; no timezone found


Not Valid Ticker (no data): AGN


$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)


Not Valid Ticker (no data): BK


$CELG: possibly delisted; no timezone found


... (5 outputs left out)

,Ticker,Stock's Name,Exchange,Currency,Ticker Type,Sector,Monthly Trading Days,Average Volume
0,AAPL,Apple Inc.,NMS,USD,EQUITY,Technology,19,4.828203e+07
1,ABBV,AbbVie Inc.,NYQ,USD,EQUITY,Healthcare,19,6.281745e+06
2,ABT,Abbott Laboratories,NYQ,USD,EQUITY,Healthcare,19,1.008423e+07
3,ACN,Accenture plc,NYQ,USD,EQUITY,Technology,19,6.116120e+06
4,AIG,"American International Group, Inc.",NYQ,USD,EQUITY,Financial Services,19,4.324195e+06
5,AMZN,"Amazon.com, Inc.",NMS,USD,EQUITY,Consumer Cyclical,19,4.652561e+07
6,AXP,American Express Company,NYQ,USD,EQUITY,Financial Services,19,3.054224e+06
7,BA,The Boeing Company,NYQ,USD,EQUITY,Industrials,19,7.140954e+06
8,BAC,Bank of America Corporation,NYQ,USD,EQUITY,Financial Services,19,3.673368e+07
9,BB.TO,BlackBerry Limited,TOR,CAD,EQUITY,Technology,19,3.406802e+06


,AAPL_Close,AAPL_Adj_Close,ABBV_Close,ABBV_Adj_Close,ABT_Close,ABT_Adj_Close,ACN_Close,ACN_Adj_Close,AIG_Close,AIG_Adj_Close,...,TXN_Close,TXN_Adj_Close,UNH_Close,UNH_Adj_Close,UNP_Close,UNP_Adj_Close,UPS_Close,UPS_Adj_Close,USB_Close,USB_Adj_Close
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,255.449997,254.316605,244.380005,235.580231,133.470001,129.656540,243.710007,236.889038,79.639999,77.725349,...,180.389999,174.982285,348.299988,339.266174,234.740005,229.894775,84.379997,78.023796,47.880001,46.106373
2025-10-02,257.130005,255.989151,236.559998,228.041824,132.990005,129.190247,244.339996,237.501373,79.639999,77.725349,...,182.320007,176.854446,353.720001,344.545624,234.520004,229.679306,85.570000,79.124161,47.910000,46.135265
2025-10-03,258.019989,256.875183,233.910004,225.487244,134.589996,130.744537,245.320007,238.453979,81.019997,79.072174,...,180.320007,174.914398,360.200012,350.857574,236.800003,231.912247,86.779999,80.243004,48.070000,46.289337
2025-10-06,256.690002,255.551102,230.190002,221.901184,133.740005,129.918839,248.169998,241.224197,83.019997,81.024086,...,181.809998,176.359695,358.769989,349.464630,237.410004,232.509674,86.620003,80.095070,47.720001,45.952305
2025-10-07,256.480011,255.342056,232.830002,224.446136,133.020004,129.219406,251.229996,244.198563,84.050003,82.029327,...,177.050003,171.742401,363.660004,354.227814,231.860001,227.074219,85.980003,79.503281,47.500000,45.740448
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-24,335.920013,335.664978,265.119995,263.502045,101.070000,100.451622,177.410004,175.998123,74.820000,74.820000,...,270.649994,269.335388,375.010010,375.010010,274.070007,274.070007,92.050003,90.706924,58.400002,57.857586
2026-09-25,341.070007,340.811066,264.339996,262.726807,101.290001,100.670280,176.110001,174.708466,74.169998,74.169998,...,278.070007,276.719360,376.589996,376.589996,273.790009,273.790009,93.959999,92.589050,59.330002,58.778946
2026-09-28,338.399994,338.143066,266.279999,264.654968,100.980003,100.362175,174.470001,173.081512,72.779999,72.779999,...,278.309998,276.958191,377.829987,377.829987,274.100006,274.100006,94.250000,92.874825,58.779999,58.234055


[*********************100%***********************]  1 of 1 completed


Date
2025-10-01    1.39235
2025-10-02    1.39395
2025-10-03    1.39609
2025-10-06    1.39650
2025-10-07    1.39460
               ...   
2026-09-24    1.41048
2026-09-25    1.41460
2026-09-28    1.41548
2026-09-29    1.41771
2026-09-30    1.41913
Name: CAD=X, Length: 256, dtype: float64

## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

In [ ]:
# Convert all USD stock prices to CAD
cad_prices = convert_usd_to_cad(stock_prices, exchange_rates, normalized_tickers)

# Get S&P 500 prices and convert them to CAD
market = create_dataframe(pd.DataFrame({"Ticker": ["^GSPC"]}), start_date, end_date)["^GSPC_Close"]
market = market.reindex(stock_prices.index)
market = market.dropna()
market = market * exchange_rates

# Weekly market returns
market_weekly = market.resample("W-FRI").last() #Makes the market data weekly instead of daily because of weekly competition
market_returns = market_weekly.pct_change().dropna()

# Initialize betas as a series to store data into (switched from dictionary last commit)
betas = pd.Series()

#Beta of each stock (Put this into the above code section for part b calculations)
for ticker in normalized_tickers["Ticker"]:

    stock_weekly = cad_prices[ticker + "_Close"].resample("W-FRI").last()
    stock_returns = stock_weekly.pct_change().dropna()
    covariance = stock_returns.cov(market_returns)
    variance = market_returns.var()
    beta = covariance / variance
    betas[ticker] = beta

betas = betas.dropna()
betas = betas.sort_values(ascending=False)
display(betas)

# Top 6 get 15%, last 4 split the remaining 10%
weight_plan = [0.15, 0.15, 0.15, 0.15, 0.15, 0.15, 0.025, 0.025, 0.025, 0.025]

# Initialize series and variables for stock weight calculation
sectors = normalized_tickers.set_index("Ticker")["Sector"]
a_weights = pd.Series()
sector_weights = pd.Series()
carry = 0

# Pick stocks from highest beta down
for ticker in betas.index:

    sector = sectors[ticker]
    current = sector_weights.get(sector, 0)
    room = round(max_weight_sector - current, 6)

    if room <= 0:
        continue

    want = weight_plan[len(a_weights)] + carry
    given = min(want, room, max_weight_each)

    a_weights[ticker] = given
    sector_weights[sector] = current + given
    carry = want - given

    if len(a_weights) == 10:
        break

print("Total Weight:", a_weights.sum())
print("Remaining Weight:", carry)

# Buy each stock with its share of the budget (fees included)
a_tickers = a_weights.index.tolist()

# Download actual purchase date prices and exchange rates
purchase_stock_prices_a = create_dataframe(normalized_tickers, purchase_date, purchase_end_date)
purchase_exchange_rates_a = get_USD_to_CAD_exchange_rate(purchase_date, purchase_end_date)
purchase_rate_a = purchase_exchange_rates_a.loc[purchase_date]

portfolio_a = pd.DataFrame(index=a_tickers, columns=["Shares"])

for ticker in a_tickers:
    price = purchase_stock_prices_a.loc[purchase_date, ticker + "_Close"]
    currency = normalized_tickers.set_index("Ticker").loc[ticker, "Currency"]

    if currency == "USD":
        price = price * purchase_rate_a

    investment = a_weights[ticker] * client_a_budget
    shares = investment / (price + purchase_rate_a * fee_per_share)
    fee = calculate_transaction_fees(shares, purchase_rate_a)

    if fee < shares * purchase_rate_a * fee_per_share:
        shares = (investment - purchase_rate_a * fee_cap_per_stock) / price

    portfolio_a.loc[ticker, "Shares"] = shares

# Show portfolio A
portfolio_a["Target Weight"] = pd.Series(a_weights)
portfolio_a["Beta"] = betas[a_tickers]
portfolio_a["Sector"] = sectors[a_tickers]

display(portfolio_a)
display(pd.Series(sector_weights, name="Sector Weight"))

# Initialize beta variable
beta_max = 0

# Calculate the weighted average of beta (combined beta)
for ticker in a_tickers:
    beta_max += a_weights[ticker] * betas[ticker]

# Calculate maximum returns
returns_max = calculate_daily_returns(calculate_portfolio_value(cad_prices, portfolio_a))

purchase_cad_prices_a = convert_usd_to_cad(purchase_stock_prices_a,purchase_exchange_rates_a,normalized_tickers)
purchase_value_a = calculate_portfolio_value(purchase_cad_prices_a, portfolio_a)

# Display basic data calculated above. (comment out during actual run)
print("Value on purchase date (CAD):", purchase_value_a.loc[purchase_date])
print("Portfolio beta: " + str(beta_max))
print("Historical daily volatility: " + str(calculate_volatility(returns_max)))
print("Historical return: " + str(calculate_portfolio_return(portfolio_a, returns_max.index[0], returns_max.index[-1])))

**IMPORTANT READ - REZA**

Reza if you're reading this. I adjusted my strategy to just pick the highest weekly beta stocks and maximize top to bottom of the top 10, and if going over, roll over into the next stock. I did this because, although a different strategy would probably be better, I thought this is the way to get full marks because I did exactly what the question told me to do, which is to maximize expected return. Therefore I had to pick the highest beta stocks in top to bottom order, adjusting for the 15% constraint and the sector cap limit. 

Let me know if this is not a good strategy for the assignment or not. - John

### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

#### Client B – Explanation
Construct a portfolio of 10–20 stocks that minimizes daily portfolio volatility while satisfying all investment constraints and staying within the $750,000 CAD budget.


**1: Data**
- Calculate the daily returns of each stock &  individual stock volatility &  covariance matrix（understand how stocks move in relation to each other）


**2: Applying the Greedy Algorithm**
- Find the one stock that has the lowest volatility.
- Add one other stock that will result in the lowest volatility together.
- Repeat the step, add in one by one and continue with the lowest volatility group.
- Result in a 10 stock portfolio.
Build an initial portfolio with relatively low volatility.


**3: Weight Optimization**
- Change the weight of the stock to not be equal.
- Try to transfer weight from one stock to another.
- From 1% change all the way to .01% change.
CHECK after:
- Portfolio volatility decreases.
- Each individual stock weight is < than 15%.
- Each sector weight < 40%.
- All stock weights > 0.
Repeat the step
Return the combination with the lowest volatility.


**4: Compare which will be the best amount of stocks 10~20**
- Apply different total stock taken number to the above two parts
- Repeat 10 stocks, 11 stocks … 20 stocks:
   - Weight Optimization
   - Applying Greedy Algorithm


**5: Stock Replacement**
- Greedy will give the best combination at the time, but not best overall
- Try to replace the stock that has not been selected, to see if volatility will go down.
- And do Weight Optimization again.


**6: Calculate final Fees**
- Already sure of the weight and stock ticker.
- Estimate transaction fees for all stocks.
- Deduct estimated fees from $750,000.
- Allocate the remaining funds according to the optimized weights.
- Calculate how many shares to purchase for each stock.
- Recalculate actual transaction fees.

In [12]:
# Convert historical stock prices in the US market into CAD price.
cad_prices = convert_usd_to_cad(stock_prices, exchange_rates, normalized_tickers)

# Filtering the ticker price dataframe and keep only the Close price.
close_columns = []

for column in cad_prices.columns:
   # Since all end with _Close, so we have to exclude the _Adj_Close.
   if column.endswith("_Close") and not column.endswith("_Adj_Close"):
       close_columns.append(column)


closing_prices_b = cad_prices[close_columns].copy()

# Remove stocks with incomplete historical price data(Due to different holidays, market close etc.
closing_prices_b = closing_prices_b.dropna(axis=1)

# Determine the daily returns
daily_returns_b = closing_prices_b.pct_change(fill_method=None)
daily_returns_b = daily_returns_b.dropna(how="all") # Delate the NaN rows for data match.

# Determine the daily volatility of each individual stock (Used to compare with each other later on)
# Returns: A series containing stock volatilities.
def calculate_individual_volatility(daily_returns):
   return daily_returns.std()
# Calculate the covariance matrix
covariance_matrix_b = daily_returns_b.cov()

# Calling functions to rank stocks from lowest to highest individual volatility
individual_volatility_b = calculate_individual_volatility(daily_returns_b)
individual_volatility_b = individual_volatility_b.sort_values()

# Determining the daily volatility of a portfolio (Portfolio Standard Deviation Formula)
# Returns: portfolio standard deviation
def calculate_portfolio_volatility(weights, covariance_matrix):
   covariance = covariance_matrix.to_numpy()
   portfolio_variance = 0
   # Calculating the sum, adding up every output, act as the sigma sign in formula.
   for i in range(len(weights)):
       for j in range(len(weights)):
           portfolio_variance += (weights[i] * weights[j] * covariance[i, j])
   # Square root of Variance to get the Standard Deviation.
   portfolio_volatility = np.sqrt(portfolio_variance)

   return portfolio_volatility

# Applying Greedy Algorithm for this stock selection
# Background information:
# Greedy Algorithm is just to get or pick the best fit everytime choosing something,
# and hopping it will end up in the best overall situation.
# Just to start building the general round 1 portfolio, since we can't assume it will end up as the best result.

# Select stocks using the greedy selection algorithm
# Returns: a list of selected stocks
def greedy_stock_selection(individual_volatility, covariance_matrix, num_stocks, stock_sectors):

   selected_stocks = []
   # Determine the maximum number of stocks per sector based on the initial equal-weight allocation.
   max_stocks_per_sector = int(num_stocks * max_weight_sector)
   # Take all the stocks name to stort in a list, for the loop.
   remaining_stocks = list(individual_volatility.index)
   # Return the stock name with the lowest individual volatility.
   first_stock = individual_volatility.idxmin()
  
   # Update the 2 list.
   selected_stocks.append(first_stock)
   remaining_stocks.remove(first_stock)

   while len(selected_stocks) < num_stocks:

       best_stock = None
       lowest_volatility = float("inf")

       for ticker in remaining_stocks:
           # Find which setor is this stock in.
           candidate_sector = stock_sectors[ticker]
           # Count the times for the stock in the same sector appear, and skip the selected stock if the sector limit is reached.
           sector_count = 0

           for stock in selected_stocks:
               if stock_sectors[stock] == candidate_sector:
                   sector_count += 1
           # Skip the stock if the sector limit is reached
           if sector_count >= max_stocks_per_sector:
               continue
           # Create a temporary list of stocks, for each time loop through
           temp_stocks = selected_stocks + [ticker]

           # Give all the stocks equal weight
           weights = [1 / len(temp_stocks)] * len(temp_stocks)

           # Get info from the covariance matrix for the temporary portfolio
           temp_covariance = covariance_matrix.loc[temp_stocks, temp_stocks]

           # Calculate portfolio volatility, call the function above.
           volatility = calculate_portfolio_volatility(weights, temp_covariance)

           # Find the stock that produces the lowest portfolio volatility
           if volatility < lowest_volatility:
               lowest_volatility = volatility
               best_stock = ticker
       # stop when there is no stock that is fit to add in.
       if best_stock is None:
           break
       # Add the best stock and remove it from the remaining one's.
       selected_stocks.append(best_stock)
       remaining_stocks.remove(best_stock)

   return selected_stocks

# Match up each stock and what sector it is from,
# used to insure we didn't go over the require weight for each sector.
stock_sectors_b = {}

for i, stock in normalized_tickers.iterrows():
   ticker = stock["Ticker"] + "_Close"
   stock_sectors_b[ticker] = stock["Sector"]

# Determine the total weight of each sector.
# Returns: a dictionary containing sector weights
def calculate_sector_weights(selected_stocks, weights, stock_sectors):
   sector_weights = {}
   # Run through the list , if they are from the same sector, then add their weight up.
   for i in range(len(selected_stocks)):
       ticker = selected_stocks[i]
       sector = stock_sectors[ticker]

       if sector in sector_weights:
           sector_weights[sector] += weights[i]
       else:
           sector_weights[sector] = weights[i]

   return sector_weights

# Checking if all the sector weights is within the givin requirements.
# (Maximum weight in one sector = 40%)
# Returns: Boolean
def check_sector_constraint(sector_weights):
   for sector in sector_weights:
       if sector_weights[sector] > max_weight_sector:
           return False

   return True

In [13]:
# Checking if the portfolio satisfies all conditions givin from the Portfolio rules and requirements.
# Returns: Boolean (T/F)
# Weight constraints during weight optimization
def check_weight_constraints(selected_stocks, weights, stock_sectors):

   # Check individual stock weights
   for weight in weights:
       if weight <= 0 or weight > max_weight_each:
           return False
   # Sector weights
   sector_weights = calculate_sector_weights(selected_stocks, weights, stock_sectors)

   return check_sector_constraint(sector_weights)
 

In [14]:
# ROUND 2: Adjusting it one more round.
# Keep changing the weight of the portfolio just to try if the volatility can be further decreased.

# Changing the weight between 2 stocks i and j, keep adjusting and mainly testing which one works better.
# Return a new list of weights.
def transfer_weight(weights, i, j, step):
   new_weights = weights.copy()
   # Reduce the weight of stock i and increase the weight of stock j
   new_weights[i] = new_weights[i] - step
   new_weights[j] = new_weights[j] + step

   return new_weights

# Test different weight assigned to reduce portfolio volatility
# Returns: a list of improved portfolio weights
def optimize_portfolio_weights(selected_stocks, initial_weights,covariance_matrix, stock_sectors,step=0.01):

   best_weights = initial_weights.copy()# keep initial weight value just for better comparation.
   best_volatility = calculate_portfolio_volatility(best_weights, covariance_matrix)
  
   # go through the whole select list of stock to check.
   for i in range(len(best_weights)):
       for j in range(len(best_weights)):
           # Only transfer weight between two different stocks
           if i != j:

               new_weights = transfer_weight(best_weights, i, j, step)
               # New check after each change time.
               if check_weight_constraints(selected_stocks, new_weights, stock_sectors):

                   new_volatility = calculate_portfolio_volatility(new_weights, covariance_matrix)
                   # Only accept the change if it makes the volatility lower than before, otherwise don't change.
                   if new_volatility < best_volatility:
                       best_weights = new_weights
                       best_volatility = new_volatility

   return best_weights

# Repeat weight optimization until no further improvement can be done.
def repeat_optimization(selected_stocks, weights,covariance_matrix, stock_sectors,step=0.01):

   while True:
       new_weights = optimize_portfolio_weights(selected_stocks, weights,covariance_matrix, stock_sectors,step)
       # Compare portfolio volatility before and after optimization
       old_volatility = calculate_portfolio_volatility(weights, covariance_matrix)
       new_volatility = calculate_portfolio_volatility(new_weights, covariance_matrix)

       # Stop when the improvement is way too small, that will have no nesseccary change.
       if old_volatility - new_volatility < 1e-10:
           return weights
      
       weights = new_weights

# Optimize portfolio weights using different step sizes, automatically adjust the stock weight by different size as needed.
# Start with larger changes, and then make smaller adjustments
def multi_step_optimization(selected_stocks, weights,covariance_matrix, stock_sectors):
   # Different step szie for better adjustion, so that we can transfer weights in steps of 1%, 0.5%, 0.1%, and 0.01%.
   step_sizes = [0.01, 0.005, 0.001, 0.0001]

   for step in step_sizes:
       weights = repeat_optimization(selected_stocks, weights,covariance_matrix, stock_sectors, step)

   return weights

In [15]:
# Find stocks that are not currently in the portfolio
def get_remaining_stocks(selected_stocks, individual_volatility):
   remaining_stocks = []

   for ticker in individual_volatility.index:
       if ticker not in selected_stocks:
           remaining_stocks.append(ticker)

   return remaining_stocks

# Find the stock that we could replace to reduces portfolio volatility
# result in a list of stocks after testing possible replacements.
def replace_stock(selected_stocks, weights, individual_volatility,covariance_matrix, stock_sectors):
   best_stocks = selected_stocks.copy() # Keep a copy for further comparison.
   best_volatility = calculate_portfolio_volatility(weights, covariance_matrix.loc[selected_stocks, selected_stocks])
   remaining_stocks = get_remaining_stocks(selected_stocks, individual_volatility)

   # Go through all the replacable functions, i represents the position of the stock to be replace out, and ticker represents the stock to be replace in.
   # i is the position of the stock being removed
   # ticker is representing the stock being added
   for i in range(len(selected_stocks)):
       for ticker in remaining_stocks:

           new_stocks = selected_stocks.copy()
           new_stocks[i] = ticker

           # Check the constraints before accepting a replacement
           if check_weight_constraints(new_stocks, weights, stock_sectors):
               # Because the stock components have changed, it is necessary to re-new the corresponding Covariance Matrix.
               new_covariance = covariance_matrix.loc[new_stocks, new_stocks]
               new_volatility = calculate_portfolio_volatility(weights, new_covariance)

               # Only keep the option with the lowest risk
               if new_volatility < best_volatility:

                   best_stocks = new_stocks
                   best_volatility = new_volatility

   return best_stocks

# Repeatedly replace stocks and optimize portfolio weights
def optimize_stock_selection(selected_stocks, weights,individual_volatility,covariance_matrix, stock_sectors):

   while True:
       # First of all, look for stock replacement plans that can reduce risks.
       new_stocks = replace_stock(selected_stocks, weights, individual_volatility,covariance_matrix, stock_sectors)
       # Determine if the composition of the stock has changed.
       if new_stocks == selected_stocks:
           return {"Stocks": selected_stocks, "Weights": weights}

       selected_stocks = new_stocks
       # Modify the weights again after changing the portfolio composition
       weights = multi_step_optimization(selected_stocks, weights,covariance_matrix.loc[selected_stocks, selected_stocks],stock_sectors)


In [16]:
# Optimize the number of stocks we should include in the portfolio, between 10 ~ 20.
# Find the portfolio size with the lowest volatility
best_volatility_b = float("inf") # positive infinity, making sure the valid portfolio risk must be lower than original.
best_stocks_b = []
best_weights_b = []
portfolio_comparison_b = []

# check different size between the range of 10~20.
for num_stocks in range(min_port_size, max_port_size + 1):

   stocks = greedy_stock_selection(individual_volatility_b, covariance_matrix_b,num_stocks, stock_sectors_b)
   # Skip the portfolio that can't be constructed.
   if len(stocks) != num_stocks:
       continue

   weights = [1 / num_stocks] * num_stocks

   if not check_weight_constraints(stocks, weights, stock_sectors_b):
       continue

   # Get the covariance matrix for the selected stocks
   covariance = covariance_matrix_b.loc[stocks, stocks]

   initial_volatility = calculate_portfolio_volatility(weights, covariance)
   # Modify the weights and calculate portfolio volatility
   weights = multi_step_optimization(stocks, weights, covariance, stock_sectors_b)
   volatility = calculate_portfolio_volatility(weights, covariance)

   # Record the results for each portfolio size
   portfolio_comparison_b.append({
       "Number of Stocks": num_stocks,
       "Initial Volatility (%)": initial_volatility * 100,
       "Optimized Volatility (%)": volatility * 100,})

   # re-new the best combanation of portfolio.
   if volatility < best_volatility_b:
       best_volatility_b = volatility
       best_stocks_b = stocks
       best_weights_b = weights

# Display the comparison of different portfolio sizes
portfolio_comparison_table_b = pd.DataFrame(portfolio_comparison_b)
display(portfolio_comparison_table_b.round(4))

# Display the best portfolio size and its volatility
if len(best_stocks_b) > 0:
   print("Best Portfolio Size:", len(best_stocks_b))
   print("Best Volatility (%):", best_volatility_b * 100)
else:
   print("No valid portfolio found.")

# Adjustion, replacement on the best portfolio
final_portfolio_b = optimize_stock_selection(best_stocks_b,best_weights_b,individual_volatility_b,covariance_matrix_b,stock_sectors_b)
# Get the final stocks and weights from the dictionary.
final_stocks_b = final_portfolio_b["Stocks"]
final_weights_b = final_portfolio_b["Weights"]

# Calculating the final Volatility
final_covariance_b = covariance_matrix_b.loc[final_stocks_b, final_stocks_b]
final_volatility_b = calculate_portfolio_volatility(final_weights_b, final_covariance_b)

# Compare volatility before and after stock replacement
replacement_comparison_b = pd.DataFrame({
   "Stage": ["Stock Before Replacement","Stock After Replacement"],
   "Daily Volatility (%)": [best_volatility_b * 100,final_volatility_b * 100]
})

display(replacement_comparison_b.round(4))
# Display the weight allocated to each sector
display(calculate_sector_weights(final_stocks_b, final_weights_b, stock_sectors_b))

,Number of Stocks,Initial Volatility (%),Optimized Volatility (%)
0,10,0.6299,0.5948
1,11,0.6355,0.5945
2,12,0.6356,0.5911
3,13,0.6385,0.5903
4,14,0.6397,0.5855
5,15,0.6350,0.5773
6,16,0.6339,0.5760
7,17,0.6361,0.5760
8,18,0.6368,0.5760
9,19,0.6397,0.5733


Best Portfolio Size: 19
Best Volatility (%): 0.5733185518637622


,Stage,Daily Volatility (%)
0,Stock Before Replacement,0.5733
1,Stock After Replacement,0.5718


{'Financial Services': 0.3097947368421051,
 'Consumer Defensive': 0.20442631578947365,
 'Communication Services': 0.14993157894736836,
 'Industrials': 0.09686315789473679,
 'Consumer Cyclical': 0.07293157894736839,
 'Healthcare': 0.11695789473684204,
 'Technology': 0.04909473684210524}

In [ ]:
# Calculation of the budget allocation and transaction fees. (Total investment fees + Total transaction fees need to be less than $750,000 CAD)
# Creating a new list for this.
final_tickers_b = []

for stock in final_stocks_b:
   ticker = stock.replace("_Close", "")
   final_tickers_b.append(ticker)

# Using the actal code running date time, 2026-11-27 close price.
purchase_stock_prices = create_dataframe(normalized_tickers, purchase_date, purchase_end_date)
purchase_exchange_rates = get_USD_to_CAD_exchange_rate(purchase_date, purchase_end_date)
purchase_cad_prices = convert_usd_to_cad(purchase_stock_prices,purchase_exchange_rates,normalized_tickers)
purchase_rate_b = purchase_exchange_rates.loc[purchase_date]

# Store the purchase price of each selected stock in CAD
purchase_prices_b = []

for stock in final_stocks_b:

   price = purchase_cad_prices.loc[purchase_date, stock]
   purchase_prices_b.append(price)

# Calculate and estimate the total transaction fees
total_fees_b = 0

for i in range(len(final_tickers_b)):

   investment = client_b_budget * final_weights_b[i]
   shares = investment / purchase_prices_b[i]
   fee = calculate_transaction_fees(shares, purchase_rate_b)
   total_fees_b += fee

# Calculate the available investment after transaction fees
available_investment_b = client_b_budget - total_fees_b

# Calculate the actual number of shares to purchase
final_shares_b = []

for i in range(len(final_tickers_b)):
   # Assigned the remaining budget based on the optimized weights
   investment = available_investment_b * final_weights_b[i]
   shares = investment / purchase_prices_b[i]
   final_shares_b.append(shares)

# Calculate again the actual transaction fees
actual_fees_b = 0

for shares in final_shares_b:

   actual_fees_b += calculate_transaction_fees(shares, purchase_rate_b)

# Display out the budget allocation and transaction fees
print("Client B Budget:", client_b_budget)
print("Estimated Transaction Fees:", total_fees_b)
print("Available Investment:", available_investment_b)
print("Actual Transaction Fees:", actual_fees_b)

# Finlize and show the final portfolio as a dataframe.
portfolio_data_b = []

for i in range(len(final_tickers_b)):

   ticker = final_tickers_b[i]
   # Getting the original closing price and trading currency.
   price = purchase_stock_prices.loc[purchase_date, final_stocks_b[i]]
   currency = normalized_tickers.set_index("Ticker").loc[ticker, "Currency"]
   # Getting the actual number of shares purchased.
   shares = final_shares_b[i]
   # Calculating the total investment value on each stock in CAD.
   value = shares * purchase_prices_b[i]

   # Store the information for each stock
   portfolio_data_b.append({
       "Ticker": ticker,
       "Price": price,
       "Currency": currency,
       "Shares": shares,
       "Value": value
   })

# Converting the list to DataFrame.
Portfolio_Final_B = pd.DataFrame(portfolio_data_b)

# Calculate the Actual Weight of each stock based on its investment value.
Portfolio_Final_B["Weight"] = (Portfolio_Final_B["Value"] /Portfolio_Final_B["Value"].sum())

# Start the DataFrame index from 1 instead of 0. (Count the stock from 1 instead of 0)
Portfolio_Final_B.index = Portfolio_Final_B.index + 1
display(Portfolio_Final_B)

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**Stock Selection and Diversification**
- Client B plans to invest CAD $750,000 and aims to achieve the lowest possible portfolio volatility while remaining fully invested in stocks.
- This means that we need to minimize the risk, while maintaining a fully invested stock portfolio.
- If we use an equal weights comparison (1/N), it will assign every stock the same weight, ignoring the fact that some stocks bounce around a lot more than others, so riskier stocks end up driving a bigger share of the portfolio's ups and downs. 
- If we only pick the quietest stocks on their own, we can easily end up piled into one sector (like utilities) and ignore how the stocks move relative to each other. This can also lead to minimal movement in the overall returns of the portfolio.
- Therefore we aim to focus mainly on reducing the standard deviation of daily portfolio returns. We are not going to consider maximizing expected returns.
- First, we calculate the historical daily returns in CAD, the individual stock volatility, and the covariance of the given stocks.
- We reference the Greedy algorithm to do the first round of stock selection which reduces the overall portfolio volatility. Considering both the individual stock risk and the benefits of the diversification.

**Portfolio Weighting and Optimization**
- After the first stock selection, we adjust the stock weights to reduce the portfolio volatility, while ensuring that we satisfy all the rules.
- We start with equal weights and gradually transfer weights between stocks using step sizes ranging from 1% to 0.01%.
- We only accept weight adjustments that reduce portfolio volatility while satisfying the maximum individual weight of 15%, maximum sector weight of 40%, and positive weight requirements.
- By comparing the performance of stock sizes ranging from 10 to 20, we select the best possible approach from it.
- The approach above makes financial sense, since the portfolio risk depends not only on individual stock volatility but also on the relative movements of stocks.
- We then perform stock replacement and further weight optimization to determine whether replacing existing stocks with other eligible stocks can further reduce portfolio risk.

**Budget Management**
- We calculate and estimate the transaction fees before allocating the available investment funds according to the optimized weights.
- Fractional shares allow us to invest almost the entire CAD $750,000 budget without borrowing, short selling, or using leverage.

Overall, the approach above makes financial sense because it applies diversification, covariance, and portfolio weighting to reduce risk while respecting Client B's investment constraints. By diversifying the risk, selecting stocks with lower covariance reduces overall risk.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.
- Standard deviation is a measure of investment risk discussed in Chapter 15.  A lower standard deviation indicates that the portfolio returns fluctuate less, which aligns with Client B's investment objective.
- We compare the initial and optimized volatility of portfolios containing 10 to 20 stocks. This allows us to evaluate the effectiveness of weight optimization and determine an appropriate portfolio size.
- We used historical daily volatility covariance, and an equal weighted benchmark to evaluate our strategy.
- After all the analysis, we compared our optimized portfolio with an equal-weighted portfolio containing the exact same stocks over the same historical period. Both portfolios are valued in Canadian dollars and have a fixed number of stocks to ensure consistency in comparison.
- When the optimized portfolio shows a lower volatility, then this indicates that our weighting strategy is effective in reducing historical portfolio risk.

Therefore, daily volatility is the primary measurement of our strategy's effectiveness.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.
- The strategy that we use mainly relies on the historical volatility and covariance, so this might not correctly reflect the future market conditions, since the future is unpredictable.
- During market downturns or unexpected economic events: We have recent events that highlight this case, CEO of Nvidia Jensen Huang’s dismissal and rehiring, the Anthropic IPO, etc. The correlations between stocks may increase, reducing the returns of diversification.
- On the other hand, because this strategy prioritizes risk reduction over potential returns, it may underperform during strong market rallies.
- Our greedy selection and iterative weight optimization methods do not guarantee a global minimum-risk portfolio.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

In [ ]:
# Equal-weight Comparison for Client B

# Calculating the equal weight portfolio
# Create an equal-weight portfolio
equal_weights_b = [1 / len(final_stocks_b)] * len(final_stocks_b)
# Using the same historical CAD prices for both portfolios
comparison_prices_b = closing_prices_b[final_stocks_b].dropna()

# Calculate historical portfolio values using fixed shares
# Returns: a Series containing daily portfolio values
def calculate_historical_values(prices, weights):

   portfolio_values = pd.Series(0.0, index=prices.index)

   for i in range(len(weights)):
       # Calculate the # of shares based on the initial weight and the closing price on the first trading day.
       # Don't need to * 750,000 since its just for a comparition, so we can just use a random price ex. $1
       # same ratio will be shown, so no need * 750,000.
       shares = weights[i] / prices.iloc[0, i]
       portfolio_values += prices.iloc[:, i] * shares

   return portfolio_values

# Calculate historical values for both portfolios
optimized_values_b = calculate_historical_values(comparison_prices_b, final_weights_b)
equal_values_b = calculate_historical_values(comparison_prices_b, equal_weights_b)

# Calculate daily portfolio returns
optimized_returns_b = calculate_daily_returns(optimized_values_b)
equal_returns_b = calculate_daily_returns(equal_values_b)

# Calculate daily portfolio volatility
optimized_risk_b = calculate_volatility(optimized_returns_b)
equal_risk_b = calculate_volatility(equal_returns_b)

# Calculating the historical return.
optimized_return_b = (optimized_values_b.iloc[-1] / optimized_values_b.iloc[0] - 1)
equal_return_b = (equal_values_b.iloc[-1] / equal_values_b.iloc[0] - 1)

# Creating a comparison table
comparison_b = pd.DataFrame({
   "Portfolio": ["Optimized", "Equal-weight"],
   "Historical Return (%)": [optimized_return_b * 100,equal_return_b * 100],
   "Daily Volatility (%)": [optimized_risk_b * 100,equal_risk_b * 100]
})
display(comparison_b.round(4))
print("Historical Period:",comparison_prices_b.index[0],"to",comparison_prices_b.index[-1])

NameError: name 'final_stocks_b' is not defined

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

In [ ]:
# Programmatic Validation for Client B.
valid_tickers_b = True

for ticker in Portfolio_Final_B["Ticker"]:

   if ticker not in normalized_tickers["Ticker"].values:
       valid_tickers_b = False

# Checking if the portfolio contains 10 to 20 stocks
valid_size_b = (min_port_size <= len(Portfolio_Final_B) <= max_port_size)
# Checking if each stock has a positive weight and does not exceed the maximum individual weight of 15%
valid_individual_weight_b = (Portfolio_Final_B["Weight"].max() <= max_weight_each + 1e-10 and Portfolio_Final_B["Weight"].min() > 0)
# Check whether all portfolio weights add up to 100%.
valid_total_weight_b = (abs(Portfolio_Final_B["Weight"].sum() - 1) < 1e-10)

# Check Sector Weight
final_sector_weights_b = calculate_sector_weights(final_stocks_b,Portfolio_Final_B["Weight"].tolist(),stock_sectors_b)
# Making sure there is no sector will exceed the maximum weight of 40%
valid_sector_weight_b = check_sector_constraint(final_sector_weights_b)
display(final_sector_weights_b)

# Calculating the total investment value and transaction fees
total_investment_b = Portfolio_Final_B["Value"].sum()
total_cost_b = total_investment_b + actual_fees_b
# Calculating the remaining cash after buying in all stocks
remaining_cash_b = client_b_budget - total_cost_b
# Check whether the total cost is within the client's budget -> $750,000
valid_budget_b = total_cost_b <= client_b_budget

valid_portfolio_b = (
   valid_tickers_b
   and valid_size_b
   and valid_individual_weight_b
   and valid_total_weight_b
   and valid_sector_weight_b
   and valid_budget_b
)

# Displaying the needed results
print("Eligible Tickers:", valid_tickers_b)
print("Valid Portfolio Size:", valid_size_b)
print("Individual Weight Constraint:", valid_individual_weight_b)
print("Total Weight Constraint:", valid_total_weight_b)
print("Sector Weight Constraint:", valid_sector_weight_b)
print("Within Budget:", valid_budget_b)
print("All Constraints Satisfied:", valid_portfolio_b)

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

In [ ]:
# Final Portfolio Output - Client B
display(Portfolio_Final_B)

# Budget Reconciliation
# Display the total investment, transaction fees and remaining cash
print("Initial Budget:", client_b_budget)
print("Total Investment:", total_investment_b)
print("Transaction Fees:", actual_fees_b)
print("Total Cost:", total_cost_b)
print("Remaining Cash:", remaining_cash_b)

# Generate CSV File only when all conditions are satisfied.
if valid_portfolio_b:
   Portfolio_Final_B[["Ticker", "Shares"]].to_csv("Stocks_Group_05_B.csv", index=False)

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.

John Mandalas

Mohammadreza Babaeizadeh

Chloe Pang

Pranav Sudev

